# Phase 4 · From mechanism to function

Phases 1–3 established that the attention sink exists, is query-driven, and that a
full query transplant reproduces the sink *pattern*. Phase 4 asks the independent
question: **does that pattern matter for what the model computes?** A sink can be
query-driven yet functionally inert (a "null attention" whose value vector is ~0),
so the load-bearing evidence here is *functional* — teacher-forced NLL, per-token
KL, and top-1 agreement — not the attention pattern.

Five experiments over one editable attention path:

| | experiment | intervention |
|---|---|---|
| 1 | Sink necessity | remove BOS attention + renormalize |
| 2 | Control ablation | remove a matched non-sink key (specificity) |
| 3 | Dose–response | scale BOS attention by γ, renormalize |
| 4 | Query intervention | project the sink direction out of the query |
| 5 | Streaming | windowed attention ± sink retention |

**Locked design decisions.** Functional metrics are reported on a **held-out corpus
(WikiText-103 validation)**; the frozen benchmark is used only for mechanistic
comparability with Phases 1–3. The sink-head set **H★ is frozen from Phase 2 and
never recomputed** on the eval corpus; the matched non-sink set **H° is chosen once
and frozen**.

## Part 0 · Setup

In [ ]:
import sys, os, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
else:
    BASE = Path('.')

for name in ('phase4_utils.py', 'phase3_utils.py'):
    for cand in [BASE, Path('/content'), Path('.')]:
        if (cand / name).exists():
            sys.path.insert(0, str(cand)); break

import phase3_utils as U
import phase4_utils as P4
import numpy as np, torch, pandas as pd
print('phase3_utils + phase4_utils loaded')

CONFIG = dict(
    MODEL_NAME='Qwen/Qwen3-1.7B',
    WINDOW_LEN=512,          # WikiText window length
    N_WINDOWS=200,           # held-out windows for functional metrics
    MIN_QUERY_POS=4,
    DTYPE='float32',         # margins/KL compared at fine scale
    STREAM_WINDOW=128,       # sliding-window size for Exp 5
    STREAM_SINK_KEEPS=(0, 1, 4),
    DOSE_GAMMAS=(0.0, 0.25, 0.5, 0.75, 1.0),
    SEED=0,
)
torch.manual_seed(CONFIG['SEED'])
OUT_DIR = BASE / 'results' / 'phase4'
(OUT_DIR / 'tables').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
print('outputs ->', OUT_DIR)

### Load Qwen3-1.7B (eager attention required)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
DTYPE = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[CONFIG['DTYPE']]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
t0 = time.time()
tok = AutoTokenizer.from_pretrained(CONFIG['MODEL_NAME'])
try:
    model = AutoModelForCausalLM.from_pretrained(CONFIG['MODEL_NAME'], dtype=DTYPE,
                                                 attn_implementation='eager')
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(CONFIG['MODEL_NAME'], torch_dtype=DTYPE,
                                                 attn_implementation='eager')
model = model.to(DEVICE).eval()
model.config._attn_implementation = 'eager'      # REQUIRED: edits patch eager path
c = model.config; top = P4.model_topology(model)
print(f"loaded in {time.time()-t0:.1f}s on {DEVICE}; attn={c._attn_implementation}")
print(f"  layers={top['n_layers']} heads={top['n_heads']} KV={top['n_kv']} n_rep={top['n_rep']}")

### Held-out corpus (WikiText-103 validation) + frozen benchmark

Functional metrics come **only** from WikiText-103 validation, chunked into fixed
512-token windows, each BOS-prefixed exactly as in Phase 3. The frozen benchmark is
loaded separately for mechanistic comparability.

`REQUIRE_REAL_CORPUS=True` makes the notebook **refuse to run on the tiny fallback**:
the fallback exists only to smoke-test the implementation, never for reported
results. If the real corpus fails to load, fix the loader (or set the flag False for
a code check only).

In [ ]:
REQUIRE_REAL_CORPUS = True     # set False ONLY for an implementation smoke-test

def bos_windows(token_ids, win, n_windows, bos_id):
    '''Chunk a long id stream into non-overlapping BOS-prefixed windows of `win`.'''
    out, step = [], win - 1        # leave room for the prepended BOS
    for s in range(0, len(token_ids) - step, step):
        ids = torch.tensor([[bos_id] + list(token_ids[s:s+step])], dtype=torch.long)
        out.append(ids.to(DEVICE))
        if len(out) >= n_windows:
            break
    return out

def load_wikitext_windows():
    '''Load WikiText-103 validation robustly. The script-based `wikitext` dataset
    breaks on recent datasets/huggingface_hub (HfUriError); the parquet mirror
    `Salesforce/wikitext` is the reliable source.'''
    from datasets import load_dataset
    attempts = [
        dict(path='Salesforce/wikitext', name='wikitext-103-raw-v1', split='validation'),
        dict(path='Salesforce/wikitext', name='wikitext-103-v1',     split='validation'),
        dict(path='wikitext', name='wikitext-103-raw-v1', split='validation', trust_remote_code=True),
    ]
    last = None
    for kw in attempts:
        try:
            ds = load_dataset(**kw)
            text = "\n\n".join(t for t in ds['text'] if t and t.strip())
            stream = tok(text, add_special_tokens=False)['input_ids']
            wins = bos_windows(stream, CONFIG['WINDOW_LEN'], CONFIG['N_WINDOWS'], c.bos_token_id)
            return wins, f"{kw['path']}:{kw['name']}"
        except Exception as e:
            last = e; print(f"  tried {kw['path']}/{kw['name']}: {type(e).__name__}")
    # last resort: read the parquet file directly from the hub
    try:
        import pandas as pd
        url = ("https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/"
               "wikitext-103-raw-v1/validation-00000-of-00001.parquet")
        df = pd.read_parquet(url)
        text = "\n\n".join(t for t in df['text'].tolist() if isinstance(t, str) and t.strip())
        stream = tok(text, add_special_tokens=False)['input_ids']
        return bos_windows(stream, CONFIG['WINDOW_LEN'], CONFIG['N_WINDOWS'], c.bos_token_id), "parquet:Salesforce/wikitext"
    except Exception as e:
        last = e
    raise RuntimeError(f"WikiText-103 validation could not be loaded: {last}")

try:
    eval_windows, corpus_name = load_wikitext_windows()
    print(f"corpus: {corpus_name}; {len(eval_windows)} windows x {CONFIG['WINDOW_LEN']} tokens")
except Exception as e:
    if REQUIRE_REAL_CORPUS:
        raise RuntimeError(
            f"Held-out corpus failed to load ({type(e).__name__}: {e}).\n"
            "Reported results REQUIRE the real corpus. Fix the loader, or set "
            "REQUIRE_REAL_CORPUS=False for an implementation smoke-test ONLY.")
    print(f"[!!] WikiText load failed ({type(e).__name__}); using SMOKE-TEST fallback.")
    passages = ["The mechanistic study of neural networks examines internal computations "
                "and the way attention distributes probability across positions."] * 20
    eval_windows = []
    for p in passages:
        ids = tok(p, return_tensors='pt', add_special_tokens=False)['input_ids']
        eval_windows.append(torch.cat([torch.tensor([[c.bos_token_id]]), ids], 1).to(DEVICE))
    corpus_name = 'FALLBACK-INVALID-FOR-RESULTS'

RESULTS_VALID = not corpus_name.startswith('FALLBACK')
if not RESULTS_VALID:
    print("[!!] corpus is a SMOKE-TEST fallback -- functional numbers below are NOT valid results.")
print(f"corpus: {corpus_name}; {len(eval_windows)} windows; "
      f"first length ~{eval_windows[0].shape[1]}; results_valid={RESULTS_VALID}")

# frozen benchmark (mechanistic comparability only)
try:
    bm = U.load_benchmark([BASE, Path('/content'), Path('.')])
    bench_spec = U.build_prompts(bm['pairs'], mode='category_block', languages=('eng', 'vie'))
    def to_ids(t):
        ids = tok(t, return_tensors='pt', add_special_tokens=False)['input_ids'][:, :255]
        return torch.cat([torch.tensor([[c.bos_token_id]]), ids], 1).to(DEVICE)
    bench_ids = [to_ids(p.text) for p in bench_spec]
    print(f"frozen benchmark: {len(bench_ids)} sequences (fingerprint {bm['fingerprint']})")
except Exception as e:
    bench_ids = []
    print(f"frozen benchmark not loaded ({type(e).__name__}); mechanistic comparison skipped")

### Frozen sink-head set H★ from the Phase-2 (layer, head) matrix

H★ is selected by **thresholding the frozen Phase-2 `layer_head_matrix.csv`** — the
same numbers Phase 2 produced, on Phase 2's own data. Thresholding is a Phase-4
selection step, not a re-derivation. H° is the matched non-sink control (matched on
layer depth), chosen once and frozen. Upload `layer_head_matrix.csv` next to the
notebook.

In [ ]:
SINK_THRESHOLD = 0.90        # sink_mass cutoff on the Phase-2 matrix (0.85~118, 0.90~77, 0.95~20 cells)

joint = P4.load_phase2_matrix(BASE / 'layer_head_matrix.csv')
SINK_HEADS = P4.select_sink_heads(joint, SINK_THRESHOLD)
NONSINK_HEADS = P4.matched_nonsink_set(SINK_HEADS, joint, seed=CONFIG['SEED'])
print(f"Phase-2 matrix: {len(joint)} (layer,head) cells, sink range "
      f"{joint.sink_mass.min():.3f}-{joint.sink_mass.max():.3f}")
print(f"H*  ({len(SINK_HEADS)} cells, sink>={SINK_THRESHOLD}):", SINK_HEADS[:8], '...')
print(f"H-circle ({len(NONSINK_HEADS)} cells):", NONSINK_HEADS[:8])
json.dump({'sink_heads': SINK_HEADS, 'nonsink_heads': NONSINK_HEADS,
           'source': 'phase2_layer_head_matrix', 'threshold': SINK_THRESHOLD,
           'corpus': corpus_name},
          open(OUT_DIR / 'frozen_head_sets.json', 'w'), indent=2)

### Transfer validation — Spearman ρ (Phase-2 vs held-out ranking)

We do **not** reselect on the held-out corpus. We only *verify* that the frozen
Phase-2 sink structure still holds on WikiText, by ranking every (layer, head) cell
on both and reporting **Spearman ρ**. High ρ ⇒ the frozen H★ transfers (rank-order
preserved), so the necessity results below are not an artifact of testing on a
different corpus. Low ρ would be a reportable finding, not a licence to reselect.

In [ ]:
# per-(layer,head) sink mass on the held-out corpus (memory-light: reduces to the
# BOS column on the fly, so it is safe on 512-token windows)
wiki_sink = P4.sink_mass_fast(model, eval_windows, min_query_pos=CONFIG['MIN_QUERY_POS'],
                              max_batches=min(50, len(eval_windows)))
rho, merged = P4.spearman_rank_agreement(joint, wiki_sink)   # joint=Phase-2, wiki=held-out
print(f"Spearman rho (Phase-2 vs WikiText sink ranking, {len(merged)} cells): {rho:.3f}")

# H* vs H-circle mean sink mass on held-out text (should stay well separated)
wl = wiki_sink.set_index(['layer','head'])['sink_mass']
m_star = float(np.nanmean([wl.get((L,h), np.nan) for (L,h) in SINK_HEADS]))
m_circ = float(np.nanmean([wl.get((L,h), np.nan) for (L,h) in NONSINK_HEADS]))
print(f"held-out sink mass: H*={m_star:.3f}  H-circle={m_circ:.3f} (expect H* >> H-circle)")
P4.plot_rank_agreement(merged, rho, OUT_DIR / 'figures' / 'transfer_spearman.png',
                       sink_pairs=SINK_HEADS, show=True)
json.dump({'spearman_rho': rho, 'holdout_sink_mass_Hstar': m_star,
           'holdout_sink_mass_Hcircle': m_circ},
          open(OUT_DIR / 'transfer_validation.json', 'w'), indent=2)

## Experiment 1 · Sink necessity

Remove BOS attention (key 0) and renormalize, under three scopes — global, H★,
and the matched control H° — and measure ΔNLL / KL / top-1 vs baseline on the
held-out corpus. Sink-specific necessity shows as **H★ ≫ H°**.

In [ ]:
d1 = P4.experiment_necessity(model, eval_windows, SINK_HEADS, NONSINK_HEADS,
                             min_query_pos=CONFIG['MIN_QUERY_POS'])
s1 = P4.summarize(d1, by=['condition'])
d1.to_csv(OUT_DIR / 'tables' / 'exp1_necessity_per_token.csv', index=False)
s1.to_csv(OUT_DIR / 'tables' / 'exp1_necessity_summary.csv', index=False)
display(s1)
P4.plot_scope_bars(s1, OUT_DIR / 'figures' / 'exp1_scope_bars.png', show=True)
P4.plot_position_curve(d1, 'kl', OUT_DIR / 'figures' / 'exp1_kl_position.png', show=True)

### Null-attention check: value norm of the sink

If removal is cheap despite large sink mass, the sink may be a *null attention* —
verify via the BOS value norm ‖v₀‖ and per-head correlation of ΔNLL with sink mass.

In [ ]:
v0 = P4.value_sink_norms(model, eval_windows[0])
display(v0.groupby('layer')['v0_norm'].mean().to_frame('mean_||v0||').T)
print('If ||v0|| is small where sink mass is large, an Exp-1 null is the null-attention regime.')

## Experiment 2 · Control ablation (specificity)

Remove BOS vs a non-sink key (fixed position 1) at the H★ scope. If BOS removal
costs more than the control at comparable removed mass, the necessity is
**sink-specific**, not generic mass-removal damage.

In [ ]:
d2 = P4.experiment_control(model, eval_windows, SINK_HEADS, NONSINK_HEADS,
                           min_query_pos=CONFIG['MIN_QUERY_POS'])
s2 = P4.summarize(d2)
d2.to_csv(OUT_DIR / 'tables' / 'exp2_control_per_token.csv', index=False)
display(s2)
P4.plot_scope_bars(s2, OUT_DIR / 'figures' / 'exp2_control_bars.png', show=True)

## Experiment 3 · Dose–response

Scale BOS attention by γ (then renormalize) at the H★ scope. γ=1 recovers baseline
(validation), γ=0 reduces to Exp 1. The curve's shape — linear vs threshold vs
saturating — reveals how the model uses the sink.

In [ ]:
d3 = P4.experiment_dose(model, eval_windows, SINK_HEADS,
                        gammas=CONFIG['DOSE_GAMMAS'], min_query_pos=CONFIG['MIN_QUERY_POS'])
d3.to_csv(OUT_DIR / 'tables' / 'exp3_dose_per_token.csv', index=False)
display(d3.groupby('gamma')[['delta_nll', 'kl', 'top1_agree']].mean().round(5))
assert abs(d3[d3.gamma == 1.0]['delta_nll'].mean()) < 1e-5, 'gamma=1 must recover baseline'
P4.plot_dose(d3, OUT_DIR / 'figures' / 'exp3_dose_response.png', show=True)

## Experiment 4 · Query-level intervention

Project the sink direction out of the query (sets the BOS logit to 0 by
construction) at the H★ scope, with functional read-outs. Overlaying this on the
Exp 3 dose curve tests whether the *query* mechanism is causally upstream of the
functional effect, not merely of the pattern.

In [ ]:
d4 = P4.experiment_query(model, eval_windows, SINK_HEADS, min_query_pos=CONFIG['MIN_QUERY_POS'])
s4 = P4.summarize(d4)
d4.to_csv(OUT_DIR / 'tables' / 'exp4_query_per_token.csv', index=False)
display(s4)
P4.plot_scope_bars(s4, OUT_DIR / 'figures' / 'exp4_query_bars.png', show=True)

## Experiment 5 · Streaming (windowed attention)

Restrict attention to a sliding window of size W, retaining `sink_keep` leading
sink tokens (StreamingLLM-style), and measure per-position NLL. If sink retention
flattens the NLL that otherwise blows up at window boundaries, the sink is
functionally critical for streaming stability.

*Approximation:* this is the dense-mask proxy — positions/RoPE are the original
ones. True KV eviction with re-based positions is a heavier extension.

In [ ]:
d5 = P4.experiment_streaming(model, eval_windows, SINK_HEADS,
                             window=CONFIG['STREAM_WINDOW'],
                             sink_keeps=CONFIG['STREAM_SINK_KEEPS'],
                             min_query_pos=CONFIG['MIN_QUERY_POS'])
d5.to_csv(OUT_DIR / 'tables' / 'exp5_streaming_per_token.csv', index=False)
display(d5.groupby('sink_keep')[['nll_int', 'delta_nll', 'kl']].mean().round(4))
P4.plot_streaming(d5, OUT_DIR / 'figures' / 'exp5_streaming.png', show=True)

## Interpretation (data-driven)

Read the experiments jointly (see the Phase-4 spec's decision logic):
- **Exp 1 null + Exp 5 positive** ⇒ sink is a streaming-stability device, inert short-range.
- **Exp 1 positive + Exp 2 specific + Exp 3 threshold + Exp 4 on-curve** ⇒ sink is
  load-bearing, sink-specific, partially redundant, and query-driven to the output.
- **Exp 4 off the Exp 3 curve** ⇒ the query has functional roles beyond the sink.

In [ ]:
summary = {
    'corpus': corpus_name,
    'results_valid': bool(RESULTS_VALID),
    'n_windows': len(eval_windows),
    'window_len': CONFIG['WINDOW_LEN'],
    'sink_threshold': SINK_THRESHOLD,
    'n_sink_heads': len(SINK_HEADS),
    'spearman_rho_phase2_vs_holdout': rho,
    'exp1_necessity': P4.summarize(d1).to_dict('records'),
    'exp2_control': P4.summarize(d2).to_dict('records'),
    'exp3_dose': d3.groupby('gamma')['delta_nll'].mean().round(5).to_dict(),
    'exp4_query': P4.summarize(d4).to_dict('records'),
    'exp5_streaming': d5.groupby('sink_keep')['delta_nll'].mean().round(5).to_dict(),
}
json.dump(summary, open(OUT_DIR / 'phase4_summary.json', 'w'), indent=2, default=float)
if not RESULTS_VALID:
    print("=" * 70)
    print("WARNING: corpus is the SMOKE-TEST fallback. These numbers are NOT")
    print("valid results. Load WikiText-103 validation before reporting anything.")
    print("=" * 70)
print(json.dumps(summary, indent=2, default=float)[:1400])

## Download

In [ ]:
import shutil, tempfile
zb = Path(tempfile.mkdtemp()) / 'phase4'
shutil.make_archive(str(zb), 'zip', OUT_DIR)
print('zipped ->', str(zb) + '.zip')
if IN_COLAB:
    from google.colab import files; files.download(str(zb) + '.zip')
else:
    print('artifacts under', OUT_DIR)